# 리서치 에이전트

In [41]:
from typing import TypedDict, List, Dict
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from typing import Literal
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.utils import formatdate
from datetime import date
import markdown

import requests
import os
from dotenv import load_dotenv
load_dotenv()


True

### LLM

In [2]:
llm = ChatOllama(
    base_url="http://host.docker.internal:11434",
    model="gemma4:31b-mlx",
    temperature=0.2,
    reasoning=True
)

In [3]:
class State(TypedDict):
    query: str
    keywords: List[str]
    articles: Dict[str, str]
    report: str
    email: str

### Keyword Extract

pydantic으로 출력 제어, email도 list로 만들면 여러 사람에게 이메일 전송 가능

In [4]:
class KeywordExtract(BaseModel):
    keywords : list[str] = Field(description="사용자의 질의에서 중요 키워드를 추출")
    email : str = Field(description="사용자의 질의에서 이메일 주소가 있으면 추출")
    
def keyword_extract_node(state : State) -> dict:
    parser = PydanticOutputParser(pydantic_object=KeywordExtract)
    keyword_prompt = PromptTemplate(
        template="사용자가 입력한 질문에서 Naver News에서 검색할 키워드 및 이메일을 추출한다. \n\n{format_instructions}\n\n어떠한 부가 설명도 하지 마십시오.\n\n텍스트: {text}",
        input_variables=["text"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )
    
    classifier_chain = keyword_prompt | llm |  parser
    result = classifier_chain.invoke(state['query'])
    return {'keywords': result.keywords, "email": result.email}

### Naver news api

In [34]:
client_id = os.getenv('client_id')
client_secret = os.getenv('client_secret')


In [6]:
query = ["삼성", "주식"]
loop = 1

In [17]:
def get_news(query: list) -> str:
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret
    }


    result = {}
    
    if type(query) == str:
        query = list(query)

    for q in query:
        article_total = "========article========\n"
        params = {'query' : q, 'display' : 10, 'start': 1, "sort" : "date"}
        article_total += "\n========article========\n".join([f"title: {x['title']}\n\ndescription: {x['description']}\npubDate: {x['pubDate']}" for x in requests.get(url, headers=headers, params=params).json()['items']])
        result[q] = article_total

    return result


In [19]:
class NewsSummary(BaseModel):
    summary: str = Field(description="뉴스의 기사의 요약본")

def news_summary_node(state: State) -> str:
    parser = PydanticOutputParser(pydantic_object=NewsSummary)
    article = get_news(state['keywords'])

    persona = """ 
    당신은 유능한 기자입니다. 제공된 뉴스 정보를 핵심 정보 누락없이 보기 좋게 
    요약하고 정리해주세요. 
    \n\n
    {context}
    \n\n
    {format_instructions}
    """ 
    summary_prompt = PromptTemplate(
        template=persona,
        input_variables=["context"],
        partial_variables={"format_instructions": parser.get_format_instructions()}
    )
    summary_chain = summary_prompt | llm | parser

    result = summary_chain.invoke({
        "context": article
    })
    return {'report': result.summary}

In [20]:
builder = StateGraph(State)

builder.add_node("K", keyword_extract_node)
builder.add_node("S", news_summary_node)

builder.add_edge(START, "K")
builder.add_edge("K", "S")
builder.add_edge("S", END)

In [21]:
agent = builder.compile()

In [52]:
inputs = {'query' : "2026년 7월 29일 경제, 산업, 기업 보고서를 뉴스를 통해서 받고 싶어"}



In [ ]:
response= agent.invoke(inputs)

In [23]:
response

{'query': '2026년 7월 29일 경제, 산업, 기업 보고서를 뉴스를 통해서 받고 싶어',
 'keywords': ['경제', '산업', '기업 보고서'],
 'report': "### [종합 뉴스 요약] 2026년 7월 29일 주요 소식\n\n#### 1. 경제 및 지역 발전\n- **지역 상생 및 지원:** 경주 '라원'의 포항·울산 시민 입장료 혜택 제공, 김해시 서민금융통합지원센터 개소 및 1인 창조기업 지원센터 전문센터 지정 등 지역 밀착형 지원이 확대되고 있습니다.\n- **정책 및 규제:** 김남준 의원이 지역 발전 저해 규제 완화 패키지 법안을 추진 중이며, 구윤철 부총리는 주택 보유 수에 따른 차등적 부동산 세제 개편을 예고했습니다.\n- **금융 시장:** 엔저 현상으로 인해 5대 은행의 엔화 환전 규모가 1년 7개월 만에 최대치를 기록했습니다.\n- **기타:** 영덕군은 한수원과 협력해 신규 원전 건설 및 에너지 산업 육성에 나섰으며, 패션 B2B 플랫폼 '딜리셔스'는 내달 코스닥 상장을 통해 글로벌 시장 공략을 계획하고 있습니다.\n\n#### 2. 산업 동향 및 기술 혁신\n- **첨단 산업:** 광주광역시의 반도체 산업 공동성장 협력, 한국선급의 3D 프린팅 기술 안전성 첫 인증(국방·조선 활용) 등 미래 산업 생태계 조성이 활발합니다.\n- **글로벌 리스크 및 계약:** 일본 구마모토 강진으로 TSMC 등 반도체 공급망에 비상이 걸린 반면, 켄코아에어로스페이스는 KAI와 에어버스 엔진 부품 10년 장기공급계약을 체결하는 성과를 거뒀습니다.\n- **인프라 및 투자:** 코람코는 호텔·렌지덴셜 자산을 5조 원 규모로 확장할 계획이며, 제주는 공공기관의 조속한 지방 이전을 촉구하고 있습니다.\n\n#### 3. 기업 동향 및 AI 전환(AX)\n- **AI 도입 가속화:** SK텔레콤이 기업용 AI 모델 'A.X K2'를 공개하며 보고서 작성 및 분석 기능을 강화했고, 삼양식품과 더존비즈온 역시 업무 효율화를 위한 AI 전환

In [35]:
def router_function(state: State) -> str:
    # 현재 State를 확인합니다.
    # 조건에 따라 문자열 라우팅 키를 반환합니다.
    if state['email'] is not None:
        return 'send'
    else:
        return 'end'


In [ ]:
# 1. 계정 정보 설정 (문서 기반)
id_ = ""  # 보내는 사람 Gmail 주소
pass_ = ''  # Gmail 앱 비밀번호 (실제 사용 시 본인 것으로 변경)


# 2. SMTP 서버 연결 (SSL 사용, 포트 465)
smtp = smtplib.SMTP_SSL("smtp.gmail.com", 465)
smtp.login(id_, pass_)


(235, b'2.7.0 Accepted')

In [55]:
def send_email_node(state: State) -> None:
    # 1. 계정 정보 설정 (문서 기반)
    id_ = "seowoong362@gmail.com"  # 보내는 사람 Gmail 주소
    pass_ = 'wknq nogf zoxe pijd'  # Gmail 앱 비밀번호 (실제 사용 시 본인 것으로 변경)
    
    # 2. SMTP 서버 연결 (SSL 사용, 포트 465)
    smtp = smtplib.SMTP_SSL("smtp.gmail.com", 465)
    smtp.login(id_, pass_)

    msg = MIMEMultipart('alternative')
    msg['To'] = 'zip235789@gmail.com'
    msg['Date'] = str(date.today())
    msg['Subject'] = f"{date.today()} 뉴스 요약 "
    msg.attach(MIMEText(markdown.markdown(state['report']), 'html'))
    
    smtp.sendmail('gen1004@yonsei.ac.kr', 'zip235789@gmail.com', msg.as_string())



In [56]:
builder = StateGraph(State)
builder.add_node("keywordExtract" ,keyword_extract_node)
builder.add_node("news_search&summary" ,news_summary_node)
builder.add_node("send_email" ,send_email_node)
builder.add_edge(START, 'keywordExtract')
builder.add_edge('keywordExtract', 'news_search&summary')
builder.add_conditional_edges(
    'news_search&summary',
    router_function,
    {
        "send": "send_email",
        "end": END,
    }
)
builder.add_edge('send_email', END)
agent = builder.compile()


In [57]:
rt = agent.invoke(inputs)

In [58]:
rt

{'query': '2026년 7월 29일 경제, 산업, 기업 보고서를 뉴스를 통해서 받고 싶어',
 'keywords': ['경제', '산업', '기업 보고서'],
 'report': "### 📢 주요 뉴스 요약 보고서 (2026년 7월 29일)\n\n#### 1. 기업 실적 및 경영 동향\n*   **삼성물산:** 2분기 영업이익 1조 320억 원을 기록하며 전년 대비 약 37% 증가, 다각화된 포트폴리오를 통해 안정적 성장세를 유지함.\n*   **HD건설기계:** 2분기 영업이익이 전년 대비 92% 급증하며 영업이익률 10%를 처음으로 돌파함.\n*   **넥센타이어:** 2분기 매출액 8,913억 원(전년 比 10.8%↑)을 기록했으며, 특히 유럽 매출이 사상 처음으로 분기 4,000억 원을 넘어섬.\n*   **SK하이닉스:** 사상 최대 실적에도 불구하고 증권가에서는 목표 주가를 하향 조정하는 추세이나, 업황 체력에는 변화가 없다는 분석이 공존함.\n*   **HD한국조선해양:** 고부가 선박 중심의 선별 수주 전략으로 실적 신기록을 경신하며 체질 개선에 성공함.\n\n#### 2. AI 및 첨단 기술 산업\n*   **산업 AI 확산:** \n    - **경상남도:** '피지컬 AI' 글로벌 1강 도약을 위해 2035년까지 21조 원 투입 예정.\n    - **SK텔레콤:** 제조·국방·바이오 적용이 가능한 독자 AI 모델 'A.X K2'(6,880억 매개변수) 공개.\n    - **건설기계산업협회:** 42억 원 규모의 건설기계 비전AI 실증 사업 착수.\n*   **반도체 및 소재:** \n    - SK하이닉스는 광주 군공항 부지 방문 및 한국원자력연구원과의 방사선 영향평가 협력을 통해 우주 산업 경쟁력을 강화함.\n    - 반도체 호황으로 소부장(소재·부품·장비) 업체들의 실적이 개선되었으나, 업계 내 양극화 문제는 과제로 남음.\n\n#### 3. 글로벌 통상 및 외교 리스크\n*   **미국 무역 압박:** 트럼프 전 대통령의 